<a href="https://colab.research.google.com/github/sppoee/image2playlist/blob/main/notebooks/01_DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Module: Data Preprocessing & Feature Engineering

**Objective:** This notebook executes the data cleaning and filtering pipeline for the Kaggle Spotify Tracks Dataset. The finalized data is optimized for multimodal semantic matching using the CLIP model.

**Key Operations:**
* **Data Ingestion:** Automated retrieval via Kaggle API.
* **Quality Control:** Removal of null values and irrelevant audio tracks (e.g., spoken-word content).
* **Deduplication & Aggregation:** Consolidation of duplicate tracks and aggregation of overlapping genres to preserve semantic richness.
* **Feature Engineering:** Construction of composite text descriptions (`clip_metadata`) to align with visual embeddings.

In [1]:
# 1. Initialize hidden directory for Kaggle API credentials
!mkdir -p ~/.kaggle

# 2. Write authentication token to access_token file
!echo "KGAT_678387807a7b827978027cd7aa8231d5" > ~/.kaggle/access_token

# 3. Configure file permissions to restrict access
!chmod 600 ~/.kaggle/access_token

# 4. Execute dataset download via Kaggle API
!kaggle datasets download -d maharshipandya/-spotify-tracks-dataset

# 5. Extract dataset (using ./ path to bypass leading dash interpretation)
!unzip -o ./-spotify-tracks-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
License(s): ODbL-1.0
100% 8.17M/8.17M [00:00<00:00, 123MB/s]

Archive:  ./-spotify-tracks-dataset.zip
  inflating: dataset.csv             


In [2]:
import pandas as pd

# Load dataset into memory, handling the unnamed index column
df = pd.read_csv('dataset.csv', index_col=0)

# Display initial dataset dimensions and schema
print("Initial Dataset Shape:", df.shape)
display(df.head())

Initial Dataset Shape: (114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
# 1. Drop rows with null values in critical attributes
df_clean = df.dropna(subset=['track_name', 'artists', 'track_genre']).copy()

# 2. Sort by popularity to prioritize high-traffic tracks during deduplication
df_clean = df_clean.sort_values('popularity', ascending=False)

# 3. Remove duplicate entities based on track name and artist combinations
df_clean = df_clean.drop_duplicates(subset=['track_name', 'artists'], keep='first')

# 4. Apply domain-specific filters
# - Exclude non-musical content (speechiness >= 0.66)
# - Set minimum popularity threshold (popularity > 20)
mask_music = df_clean['speechiness'] < 0.66
mask_popular = df_clean['popularity'] > 20
df_filtered = df_clean[mask_music & mask_popular].copy()

# 5. Construct composite text features for downstream CLIP embedding
df_filtered['clip_metadata'] = (
    "A track titled '" + df_filtered['track_name'] +
    "' by " + df_filtered['artists'] +
    ". The vibe and genre is " + df_filtered['track_genre'] + "."
)

# Output summary metrics
print(f"Initial row count: {len(df)}")
print(f"Cleaned row count: {len(df_filtered)}")
display(df_filtered[['track_name', 'artists', 'popularity', 'clip_metadata']].head())

Initial row count: 114000
Cleaned row count: 61670


,track_name,artists,popularity,clip_metadata
20001,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100,A track titled 'Unholy (feat. Kim Petras)' by ...
51664,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,99,"A track titled 'Quevedo: Bzrp Music Sessions, ..."
88410,La Bachata,Manuel Turizo,98,A track titled 'La Bachata' by Manuel Turizo. ...
30003,I'm Good (Blue),David Guetta;Bebe Rexha,98,A track titled 'I'm Good (Blue)' by David Guet...
68304,Tití Me Preguntó,Bad Bunny,97,A track titled 'Tití Me Preguntó' by Bad Bunny...


In [4]:
# 1. Isolate all instances of duplicated tracks
duplicates_df = df[df.duplicated(subset=['track_name', 'artists'], keep=False)]

# 2. Sort and filter columns for visual inspection of duplicate patterns
view_cols = ['track_name', 'artists', 'track_genre', 'album_name', 'popularity']
duplicates_sorted = duplicates_df[view_cols].sort_values(
    by=['track_name', 'popularity'],
    ascending=[True, False]
)

# 3. Display duplication summary
print(f"Total rows identified as duplicates: {len(duplicates_sorted)}")
display(duplicates_sorted.head(20))

Total rows identified as duplicates: 49157


,track_name,artists,track_genre,album_name,popularity
93397,"""Don Carlos"" Roderigo'S Death Aria",Nikolay Kopylov,romance,Popular Opera Arias,0
93440,"""Don Carlos"" Roderigo'S Death Aria",Nikolay Kopylov,romance,Popular Opera Arias,0
16385,"""Hark! The Herald Angels Sing""",Felix Mendelssohn;Christopher Herrick;Simon Pr...,classical,Klassische Weihnachtsmusik,0
39446,"""Hark! The Herald Angels Sing""",Felix Mendelssohn;Christopher Herrick;Simon Pr...,german,Klassische Weihnachtsmusik,0
111315,"""Was He Slow?"" - Music From The Motion Picture...",Kid Koala,trip-hop,"""Was He Slow?"" (Music From The Motion Picture ...",37
111402,"""Was He Slow?"" - Music From The Motion Picture...",Kid Koala,trip-hop,Baby Driver (Music from the Motion Picture),25
4152,#3,Aphex Twin,ambient,Selected Ambient Works Volume II,64
32514,#3,Aphex Twin,electronic,Selected Ambient Works Volume II,64
61045,#おふしょるにっと,≠ME,j-idol,チョコレートメランコリー<Special Edition>,33
63939,#おふしょるにっと,≠ME,j-rock,チョコレートメランコリー<Special Edition>,33


In [5]:
# 1. Handle missing values and sort by priority metric (popularity)
df_clean = df.dropna(subset=['track_name', 'artists', 'track_genre']).copy()
df_clean = df_clean.sort_values('popularity', ascending=False)

# 2. Feature Aggregation: Consolidate overlapping genres for identical tracks
df_clean['merged_genres'] = df_clean.groupby(['track_name', 'artists'])['track_genre'].transform(lambda x: ', '.join(x.unique()))

# 3. Deduplication: Retain the primary record containing the aggregated genre data
df_clean = df_clean.drop_duplicates(subset=['track_name', 'artists'], keep='first')

# 4. Apply content and quality filters
mask_music = df_clean['speechiness'] < 0.66
mask_popular = df_clean['popularity'] > 20
df_filtered = df_clean[mask_music & mask_popular].copy()

# 5. Construct enriched text features utilizing aggregated genres
df_filtered['clip_metadata'] = (
    "A track titled '" + df_filtered['track_name'] +
    "' by " + df_filtered['artists'] +
    ". The vibe and genres are " + df_filtered['merged_genres'] + "."
)

# Validate aggregation logic (Example: Aphex Twin)
print("Validation - Genre Aggregation Result:")
display(df_filtered[df_filtered['track_name'] == '#3'][['track_name', 'artists', 'merged_genres', 'clip_metadata']])

Validation - Genre Aggregation Result:


,track_name,artists,merged_genres,clip_metadata
32514,#3,Aphex Twin,"electronic, ambient",A track titled '#3' by Aphex Twin. The vibe an...


In [6]:
# Display final schema attributes
print("Final Dataset Columns:")
print(df_filtered.columns.tolist())

Final Dataset Columns:
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'merged_genres', 'clip_metadata']


In [7]:
# Export finalized dataset to CSV format
df_filtered.to_csv('spotify_cleaned_final.csv', index=False)